# 🌀 RayCash — Entraînement du classifieur de déchets

Ce notebook **doit tourner sur Google Colab avec GPU activé** :
`Exécution → Modifier le type d'exécution → Type de matériel = T4 GPU`.

**Pipeline complet** :
1. Télécharger TrashNet (~2.5k images, 6 classes)
2. Remapper les classes vers nos 5 cibles : Aluminium, Plastique, Verre, Papier, Carton (+ Inconnu)
3. Split train / val / test
4. Fine-tuning de MobileNetV2 (pré-entraîné sur ImageNet)
5. Évaluation + matrice de confusion
6. Export TFLite **int8 quantized** (compatible avec `server/inference.py` tel quel)
7. Téléchargement de `model.tflite` + `labels.txt` à brancher dans `server/models/`

Durée estimée : ~15 min sur GPU T4.

## 0️⃣ Setup

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU dispo :', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Active le GPU dans Exécution → Modifier le type d\'exécution'

In [ ]:
import os, json, shutil, random, pathlib, zipfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

WORK_DIR = pathlib.Path('/content/raycash')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

## 1️⃣ Télécharger TrashNet

Source : https://github.com/garythung/trashnet (Creative Commons).

In [ ]:
DATASET_URL = 'https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip'
DATASET_ZIP = WORK_DIR / 'trashnet.zip'
DATASET_DIR = WORK_DIR / 'dataset-resized'

if not DATASET_DIR.exists():
    print('Téléchargement TrashNet...')
    urllib.request.urlretrieve(DATASET_URL, DATASET_ZIP)
    print('Extraction...')
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(WORK_DIR)

# TrashNet a 6 classes : cardboard, glass, metal, paper, plastic, trash
for d in sorted(DATASET_DIR.iterdir()):
    if d.is_dir():
        print(f'  {d.name}: {len(list(d.glob("*.jpg")))} images')

## 2️⃣ Remapper vers nos 5 classes RayCash

| TrashNet | RayCash | Raison |
|---|---|---|
| `cardboard` | `Carton` | direct |
| `glass` | `Verre` | direct |
| `metal` | `Aluminium` | TrashNet metal = canettes alu majoritairement |
| `paper` | `Papier` | direct |
| `plastic` | `Plastique` | direct |
| `trash` | `Inconnu` | catégorie poubelle = non recyclable |

In [ ]:
CLASS_MAPPING = {
    'cardboard': 'Carton',
    'glass': 'Verre',
    'metal': 'Aluminium',
    'paper': 'Papier',
    'plastic': 'Plastique',
    'trash': 'Inconnu',
}
# Ordre stable pour labels.txt — DOIT correspondre à l'ordre de softmax.
RAYCASH_CLASSES = ['Aluminium', 'Carton', 'Inconnu', 'Papier', 'Plastique', 'Verre']

SPLIT_DIR = WORK_DIR / 'split'
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

TRAIN_RATIO, VAL_RATIO = 0.8, 0.1  # reste = test

for src_class_name in CLASS_MAPPING:
    target = CLASS_MAPPING[src_class_name]
    src_dir = DATASET_DIR / src_class_name
    files = sorted(src_dir.glob('*.jpg'))
    random.Random(SEED).shuffle(files)
    n = len(files)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    splits = {
        'train': files[:n_train],
        'val':   files[n_train:n_train+n_val],
        'test':  files[n_train+n_val:],
    }
    for split_name, items in splits.items():
        dest = SPLIT_DIR / split_name / target
        dest.mkdir(parents=True, exist_ok=True)
        for f in items:
            shutil.copy(f, dest / f.name)

for split in ['train', 'val', 'test']:
    total = 0
    print(f'\n[{split}]')
    for cls in RAYCASH_CLASSES:
        n = len(list((SPLIT_DIR / split / cls).glob('*.jpg')))
        total += n
        print(f'  {cls:12} {n}')
    print(f'  {"TOTAL":12} {total}')

## 3️⃣ Pipelines tf.data

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

def make_dataset(split, augment=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        SPLIT_DIR / split,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_names=RAYCASH_CLASSES,  # Ordre déterministe
        shuffle=(split == 'train'),
        seed=SEED,
    )
    return ds

train_raw = make_dataset('train', augment=True)
val_raw   = make_dataset('val')
test_raw  = make_dataset('test')

print('Classes :', train_raw.class_names)
assert train_raw.class_names == RAYCASH_CLASSES

# Augmentation appliquée en runtime sur le train uniquement
data_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(lambda x, y: (data_augment(x, training=True), y),
                          num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_raw.prefetch(AUTOTUNE)
test_ds  = test_raw.prefetch(AUTOTUNE)

## 4️⃣ Modèle — Transfer learning MobileNetV2

In [ ]:
NUM_CLASSES = len(RAYCASH_CLASSES)

base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base.trainable = False  # On gèle d'abord le backbone

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

### Phase 1 — Tête seule (10 epochs)

In [ ]:
es = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)
rlr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)

history_head = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[es, rlr])

### Phase 2 — Fine-tuning du backbone (10 epochs, LR plus bas)

In [ ]:
base.trainable = True
# Garde les premières couches gelées pour ne pas casser les features bas niveau
for layer in base.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[es, rlr])

## 5️⃣ Évaluation sur test set

In [ ]:
loss, acc = model.evaluate(test_ds)
print(f'Test accuracy : {acc*100:.2f}%')

y_true, y_pred = [], []
for batch_x, batch_y in test_ds:
    probs = model.predict(batch_x, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1))
    y_true.extend(batch_y.numpy())

print(classification_report(y_true, y_pred, target_names=RAYCASH_CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=RAYCASH_CLASSES, yticklabels=RAYCASH_CLASSES)
plt.xlabel('Prédit'); plt.ylabel('Vérité')
plt.title('Matrice de confusion — RayCash classifier')
plt.tight_layout()
plt.show()

## 6️⃣ Export TFLite **int8 quantized**

On utilise post-training integer quantization avec un dataset représentatif. Le modèle exporté a :
- Input : `uint8 [1, 224, 224, 3]` ← compatible avec `server/inference.py` tel quel
- Output : `uint8 [1, 6]` ← idem
- Taille ~900 KB (vs ~14 MB en float)

In [ ]:
def representative_dataset():
    # 100 échantillons aléatoires du train pour calibrer la quantization
    for images, _ in train_raw.take(100 // BATCH_SIZE + 1):
        for img in images:
            yield [tf.cast(tf.expand_dims(img, 0), tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()
OUT_TFLITE = WORK_DIR / 'model.tflite'
OUT_TFLITE.write_bytes(tflite_model)
print(f'Modèle exporté : {OUT_TFLITE} ({OUT_TFLITE.stat().st_size / 1024:.0f} KB)')

# labels.txt — UN label par ligne, ordre identique à class_names
OUT_LABELS = WORK_DIR / 'labels.txt'
OUT_LABELS.write_text('\n'.join(f'{i} {name}' for i, name in enumerate(RAYCASH_CLASSES)))
print('Labels :')
print(OUT_LABELS.read_text())

## 7️⃣ Sanity check du TFLite

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(OUT_TFLITE))
interpreter.allocate_tensors()
in_det = interpreter.get_input_details()[0]
out_det = interpreter.get_output_details()[0]
print('Input  :', in_det['shape'], in_det['dtype'])
print('Output :', out_det['shape'], out_det['dtype'])

correct = 0
total = 0
for batch_x, batch_y in test_ds.unbatch().batch(1).take(100):
    x = tf.cast(batch_x, tf.uint8).numpy()
    interpreter.set_tensor(in_det['index'], x)
    interpreter.invoke()
    out = interpreter.get_tensor(out_det['index'])[0]
    pred = int(np.argmax(out))
    total += 1
    correct += int(pred == int(batch_y.numpy()[0]))
print(f'TFLite accuracy sur 100 échantillons test : {correct}/{total} ({correct/total*100:.1f}%)')

## 8️⃣ Télécharger les artefacts

Après exécution, copie `model.tflite` et `labels.txt` dans `server/models/` du projet RayCash.

In [ ]:
from google.colab import files
files.download(str(OUT_TFLITE))
files.download(str(OUT_LABELS))